# Decoding analysis — model 31, smoothed, spherical_noise, fit_responses, n_voxels=0

- **rm_corr / overall MAE**: full (uncensored) posteriors for both conditions
- **Narrow vs Wide MAE comparison**: wide posterior censored at x=25 and restricted to x≤25 trials, so both conditions are integrated over the same range

In [1]:
import pandas as pd
import numpy as np
import pingouin as pg
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
from pathlib import Path
from tqdm.notebook import tqdm
from neural_priors.utils.data import get_all_subject_ids

bids_folder = Path('/data/ds-neuralpriors')
decoding_key = 'model31.smoothed.spherical_noise.fit_responses'
n_voxels = 0
NARROW_UPPER = 25.0

In [2]:
def get_pdf(subject):
    target_dir = bids_folder / 'derivatives' / 'decoding2' / decoding_key / f'sub-{subject}' / 'func'
    fname = target_dir / f'sub-{subject}_mask-NPCr_nvoxels-{n_voxels}_pars.tsv'
    pdf = pd.read_csv(fname, sep='\t', index_col=[0, 1, 2, 3, 4], header=[0, 1])
    pdf.columns.names = ['n', 'range']
    pdf.columns = pd.MultiIndex.from_arrays(
        [pdf.columns.get_level_values(0).astype(np.float32),
         pdf.columns.get_level_values(1)],
        names=pdf.columns.names)
    return pdf.sort_index(level=['session', 'run', 'trial_nr'])


def _integrate(p):
    """Normalise p and return (E, SD) via numerical integration."""
    cols = p.columns.astype(float).values
    p = p.div(np.trapz(p, cols, axis=1), axis=0)
    E  = np.trapz(p * cols[np.newaxis, :], cols, axis=1)
    SD = np.trapz(np.abs(E[:, np.newaxis] - cols[np.newaxis, :]) * p, cols, axis=1)
    return E, SD


def compute_E(pdf, censor_wide=False):
    """Compute posterior mean E and SD per trial.
    censor_wide=True: truncate wide PDF at NARROW_UPPER before renormalising.
    """
    pdf0 = pdf.xs('0.0', level=1, axis=1).loc[:, :NARROW_UPPER]
    E0, SD0 = _integrate(pdf0)

    pdf1 = pdf.xs('1.0', level=1, axis=1)
    if censor_wide:
        pdf1 = pdf1.loc[:, :NARROW_UPPER]
    E1, SD1 = _integrate(pdf1)

    is_narrow = pdf.index.get_level_values('range') == 0.0
    E  = np.where(is_narrow, E0,  E1)
    SD = np.where(is_narrow, SD0, SD1)

    pars = pd.DataFrame({'E': E, 'SD': SD}, index=pdf.index)
    pars.reset_index(['x', 'range'], inplace=True)
    pars['range'] = pars['range'].map({0.0: 'narrow', 1.0: 'wide'})
    pars['error'] = pars['E'] - pars['x']
    pars['abs_error'] = pars['error'].abs()
    return pars

In [3]:
subjects = get_all_subject_ids()

pars_full_list     = []  # uncensored wide — for rm_corr and overall MAE
pars_censored_list = []  # wide censored at 25 — for narrow vs wide MAE comparison

for subject in tqdm(subjects):
    try:
        pdf = get_pdf(subject)
        pars_full_list.append(compute_E(pdf, censor_wide=False).assign(subject=subject))
        pars_censored_list.append(compute_E(pdf, censor_wide=True).assign(subject=subject))
    except FileNotFoundError:
        print(f'Missing: sub-{subject}')

pars          = pd.concat(pars_full_list)
pars_censored = pd.concat(pars_censored_list)
print(f'Subjects: {pars["subject"].nunique()}   Trials: {len(pars)}')

  0%|          | 0/39 [00:00<?, ?it/s]

Subjects: 39   Trials: 18720


## 1. rm_corr — full posteriors (uncensored)

In [4]:
rm_combined = pg.rm_corr(pars.reset_index(), x='E', y='x', subject='subject')
r_c, dof_c, p_c = rm_combined['r'].values[0], rm_combined['dof'].values[0], rm_combined['pval'].values[0]
print(f'Combined: r_rm({dof_c:.0f}) = {r_c:.3f}, P = {p_c:.2e}')

for cond in ['narrow', 'wide']:
    rm = pg.rm_corr(pars[pars['range']==cond].reset_index(), x='E', y='x', subject='subject')
    r, dof, pval = rm['r'].values[0], rm['dof'].values[0], rm['pval'].values[0]
    print(f'{cond.capitalize()}: r_rm({dof:.0f}) = {r:.3f}, P = {pval:.2e}')
    if cond == 'narrow':
        r_n, dof_n, p_n = r, dof, pval
    else:
        r_w, dof_w, p_w = r, dof, pval

Combined: r_rm(18615) = 0.341, P = 0.00e+00
Narrow: r_rm(9277) = 0.210, P = 4.37e-93
Wide: r_rm(9298) = 0.189, P = 2.79e-75


## 2. Overall MAE per condition — full posteriors

In [5]:
for cond in ['narrow', 'wide']:
    mae = pars[pars['range']==cond].groupby('subject')['abs_error'].mean()
    print(f'{cond.capitalize()} MAE (full posterior): mean={mae.mean():.2f}, SD={mae.std():.2f}')

Narrow MAE (full posterior): mean=4.83, SD=0.37
Wide MAE (full posterior): mean=9.24, SD=0.95


## 3. MAE comparison: Narrow vs Wide — wide censored at 25, restricted to x ≤ 25
Both conditions now integrate over the same range for a fair comparison.

In [6]:
narrow_mae = pars_censored[pars_censored['range'] == 'narrow'].groupby('subject')['abs_error'].mean()
wide_mae   = (pars_censored[(pars_censored['range'] == 'wide') & (pars_censored['x'] <= NARROW_UPPER)]
              .groupby('subject')['abs_error'].mean())

print(f'Narrow MAE:         mean={narrow_mae.mean():.2f}, SD={narrow_mae.std():.2f}')
print(f'Wide [<=25] MAE:    mean={wide_mae.mean():.2f}, SD={wide_mae.std():.2f}')

ttest = pg.ttest(wide_mae, narrow_mae, paired=True, alternative='greater')
T, dof_t, p_t = ttest['T'].values[0], ttest['dof'].values[0], ttest['p-val'].values[0]
print(f'Paired t-test: t({dof_t:.0f}) = {T:.2f}, P = {p_t:.2e}')

Narrow MAE:         mean=4.83, SD=0.37
Wide [<=25] MAE:    mean=4.96, SD=0.44
Paired t-test: t(38) = 1.91, P = 3.20e-02


## Copy-paste summary for manuscript

## 4. Decoded uncertainty (posterior SD) — censored posteriors, x ≤ 25
Larger posterior SD = the decoder is less certain = brain representation is less precise.

In [7]:
narrow_sd = pars_censored[pars_censored['range'] == 'narrow'].groupby('subject')['SD'].mean()
wide_sd   = (pars_censored[(pars_censored['range'] == 'wide') & (pars_censored['x'] <= NARROW_UPPER)]
             .groupby('subject')['SD'].mean())

print(f'Narrow posterior SD:       mean={narrow_sd.mean():.2f}, SD={narrow_sd.std():.2f}')
print(f'Wide [<=25] posterior SD:  mean={wide_sd.mean():.2f}, SD={wide_sd.std():.2f}')

ttest_sd = pg.ttest(wide_sd, narrow_sd, paired=True, alternative='greater')
T_sd, dof_sd, p_sd = ttest_sd['T'].values[0], ttest_sd['dof'].values[0], ttest_sd['p-val'].values[0]
print(f'Paired t-test (Wide > Narrow): t({dof_sd:.0f}) = {T_sd:.2f}, P = {p_sd:.3e}')

Narrow posterior SD:       mean=0.66, SD=0.37
Wide [<=25] posterior SD:  mean=0.78, SD=0.36
Paired t-test (Wide > Narrow): t(38) = 3.88, P = 2.005e-04


In [8]:
for method in ['pearson', 'spearman']:
    narrow_r = (pars_censored[pars_censored['range'] == 'narrow']
                .groupby('subject')
                .apply(lambda d: pg.corr(d['E'], d['x'], method=method)['r'].values[0]))

    wide_r = (pars_censored[(pars_censored['range'] == 'wide') & (pars_censored['x'] <= NARROW_UPPER)]
              .groupby('subject')
              .apply(lambda d: pg.corr(d['E'], d['x'], method=method)['r'].values[0]))

    ttest_r = pg.ttest(narrow_r, wide_r, paired=True, alternative='greater')
    T_r, dof_r, p_r = ttest_r['T'].values[0], ttest_r['dof'].values[0], ttest_r['p-val'].values[0]
    print(f'{method.capitalize()}: narrow={narrow_r.mean():.3f}±{narrow_r.std():.3f}, '
          f'wide={wide_r.mean():.3f}±{wide_r.std():.3f}, '
          f't({dof_r:.0f})={T_r:.2f}, P={p_r:.3f}')

# Save for summary
narrow_r_pearson = (pars_censored[pars_censored['range'] == 'narrow']
                    .groupby('subject')
                    .apply(lambda d: pg.corr(d['E'], d['x'])['r'].values[0]))
wide_r_pearson   = (pars_censored[(pars_censored['range'] == 'wide') & (pars_censored['x'] <= NARROW_UPPER)]
                    .groupby('subject')
                    .apply(lambda d: pg.corr(d['E'], d['x'])['r'].values[0]))
ttest_r = pg.ttest(narrow_r_pearson, wide_r_pearson, paired=True, alternative='greater')
T_r, dof_r, p_r = ttest_r['T'].values[0], ttest_r['dof'].values[0], ttest_r['p-val'].values[0]

Pearson: narrow=0.207±0.110, wide=0.191±0.138, t(38)=0.68, P=0.250


Spearman: narrow=0.212±0.111, wide=0.186±0.137, t(38)=1.12, P=0.134


In [9]:
print('=== MANUSCRIPT STATS ===')
print(f'combined:           r_rm({dof_c:.0f}) = {r_c:.3f}, P = {p_c:.2e}')
print(f'narrow:             r_rm({dof_n:.0f}) = {r_n:.3f}, P = {p_n:.2e}')
print(f'wide:               r_rm({dof_w:.0f}) = {r_w:.3f}, P = {p_w:.2e}')
print(f'narrow MAE (censored):          mean={narrow_mae.mean():.2f}, SD={narrow_mae.std():.2f}')
print(f'wide [<=25] MAE (censored):     mean={wide_mae.mean():.2f}, SD={wide_mae.std():.2f}')
print(f'MAE paired t-test:              t({dof_t:.0f}) = {T:.2f}, P = {p_t:.2e}')
print(f'narrow r (censored):            mean={narrow_r.mean():.3f}, SD={narrow_r.std():.3f}')
print(f'wide [<=25] r (censored):       mean={wide_r.mean():.3f}, SD={wide_r.std():.3f}')
print(f'r paired t-test (N>W):          t({dof_r:.0f}) = {T_r:.2f}, P = {p_r:.3f}')

=== MANUSCRIPT STATS ===
combined:           r_rm(18615) = 0.341, P = 0.00e+00
narrow:             r_rm(9277) = 0.210, P = 4.37e-93
wide:               r_rm(9298) = 0.189, P = 2.79e-75
narrow MAE (censored):          mean=4.83, SD=0.37
wide [<=25] MAE (censored):     mean=4.96, SD=0.44
MAE paired t-test:              t(38) = 1.91, P = 3.20e-02
narrow r (censored):            mean=0.212, SD=0.111
wide [<=25] r (censored):       mean=0.186, SD=0.137
r paired t-test (N>W):          t(38) = 0.68, P = 0.250
